# Create a custom cut in the notebook

This example defines a new dataset cut directly in the notebook, applies it while selecting entries, and then runs the first passing event through a small pipeline.


## Bootstrap project imports


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate reco_algorithm_tests project root.")

src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Project root: {PROJECT_ROOT}")


Project root: /workdir/playground/reco_algorithm_tests


## Import data-loading and pipeline tools


In [2]:
from algorithms import DefaultTrackletFormer
from data_io import DataCut, RecoDataFile, load_pioneer_libraries
from pipeline.pipeline import Pipeline
from pipeline.stages import EventInitStage, EventInputStage, InputContext, TrackletStage


## Define a new cut inline


In [3]:
class MinimumAtarHitsCut(DataCut):
    def __init__(self, min_hits: int = 8, *, enabled: bool = True):
        super().__init__(name="minimum_atar_hits", enabled=enabled)
        self.min_hits = int(min_hits)

    def accepts(self, data_file, entry: dict) -> bool:
        hits = entry.get("hits")
        return hits is not None and int(hits.size()) >= self.min_hits


## Open the data file with the custom cut enabled


In [4]:
load_pioneer_libraries()

DATA_PATH = PROJECT_ROOT / ".data" / "current" / "all_rec.root"
reco_data = RecoDataFile(
    str(DATA_PATH),
    cuts=[MinimumAtarHitsCut(min_hits=8)],
)

selected_entries = reco_data.selected_entries(max_entries=200)
print("First selected entries:", selected_entries[:10])
reco_data.print_cut_table()


First selected entries: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
cut                         removed  removed %   cumulative   cumulative %
--------------------------------------------------------------------------
minimum_atar_hits                 9      4.50%            9          4.50%
--------------------------------------------------------------------------
TOTAL                             9      4.50%            9          4.50%
KEPT                            191     95.50%          191         95.50%
Scanned entries: 200


## Use the selected entries in a pipeline


In [5]:
if not selected_entries:
    raise RuntimeError("The custom cut removed every entry in the tested range.")

pipeline = Pipeline()
pipeline.register_stage(EventInputStage())
pipeline.register_stage(EventInitStage())
pipeline.register_stage(TrackletStage(DefaultTrackletFormer()))

entry_index = selected_entries[0]
pipeline.run(InputContext(reco_data, entry_index))
event = pipeline.get_event()
print("Chosen entry:", entry_index)
print("ATAR hits:", len(event.all_hits))
print("Tracklets:", len(event.all_tracklets))
event


Chosen entry: 0
ATAR hits: 33
Tracklets: 2


Event(id=0, no patterns, 2 tracklets (sample IDs: [1, 0]), no vertices, 33 hits, extra_info: 2 keys: ['geo', 'tracklet_algorithm_info'], is_valid=False)